## Rare codon patch search pipeline

The main pipeline for the identification of rare codon patches in protein-coding genes.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import gzip 
from Bio import SeqIO
import math
from collections import Counter
from scipy.spatial import cKDTree
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy
from scipy.stats import gaussian_kde

In [ ]:

# loads required files for this

import yaml
from pathlib import Path
import pandas as pd

# cd to repo
# os.chdir('/Users/jacobfine/Library/CloudStorage/OneDrive-Personal/U of T 2022-2023/Blencowe/Jan_2024_Blencowe/NOV_2024_reanalysis/sequence_analysis_may_2026/compare_backgrounds_codon')


## root path to cloned repo; wherever repo was cloned
os.chdir(Path('./codon_analysis'))
print(f"Working directory: {os.getcwd()}")


class ConfigLoader:
    
    def __init__(self, config_path='read_files.yaml'):
        with open(config_path, 'r') as f:
            self.config = yaml.safe_load(f)
    
    def load_csv(self, category, key):
        # function to load csv from configs
        file_info = self.config['input_files'][category][key]
        path = Path(file_info['path'])
        csv_config = self.config['loading_config']['csv']
        
        df = pd.read_csv(path, **csv_config)
        print(f"loaded {key}: {df.shape}")
        return df
    
    def load_bed(self, category, key):
        # function to load bed file from configs
        file_info = self.config['input_files'][category][key]
        path = Path(file_info['path'])
        bed_config = self.config['loading_config']['bed']
        
        df = pd.read_csv(path, **bed_config)
        print(f"loaded {key}: {df.shape}")
        return df

loader = ConfigLoader('configs/read_files/read_files.yaml')

# uses config to load csv or bed
longest_orfs_df = loader.load_csv('seq_process', 'longest_orfs')
longest_orfs_df_expanded = loader.load_csv('seq_process', 'longest_orfs_expanded')
df_master = loader.load_csv('kmers', 'gene_master')


Working directory: /Users/jacobfine/Library/CloudStorage/OneDrive-Personal/U of T 2022-2023/Blencowe/Jan_2024_Blencowe/NOV_2024_reanalysis/sequence_analysis_may_2026/compare_backgrounds_codon
loaded longest_orfs: (20202, 6)
loaded longest_orfs_expanded: (199904, 10)
loaded gene_master: (482, 390)


In [4]:

# function to generate sliding winodws of k-mers from human ORFs. 
# A key component of this task is to jointly obtain the genomic coordinates AND sequence as the window moves

# To run this, we require a list of genes with this longest ORFs, as well as a row for each ORF segment (since they map to discontiguous coordinates in the genome)
# Each slice of an ORF is assigned a strand, which should be the same as the host ORF 
# this is the 'input_df'
def generate_kmers_and_coords(input_df, k=45, offset=3):
    results = []  # stores the results of the sliding window
    
    # first groups df by gene. Each gene corresponds to one ORF (longest ORF) and a list of genomic coordinates its segments span
    grouped = input_df.groupby("gene")

    for gene, group in grouped:  # iterates through each group
        # we parse coordinates differently by strand, and sort accordingly. For reverse strand, sort from largest to smallest.
        if group["strand"].iloc[0] == "-":
            group = group.sort_values(by=["end"], ascending=False)  # sort values by the end coordinate from highest to lowest values
        else:
            group = group.sort_values(by=["start"])  # otherwise go from smallest to largest
        
        # get the full sequence of the ORF
        full_sequence = group["assembled_ORF"].iloc[0]
        positions = []  # each ORF has a list of segments, which due to splicing are not continuous in terms of genomic coordinates.

        # iterates through each 4-tuple representing chromosome coordinates (chr, start, end, strand)
        for _, row in group.iterrows():
            positions.append((row["chromosome"], row["start"], row["end"], row["strand"]))  # we update the list of coordiantes for that orf
        
        # now, we perform the sliding window accross the ORF sequence, while also getting the cooresponding coordinates.
        for start_index in range(0, len(full_sequence) - k + 1, offset):  # we keep track of the start index since that is relative to the host ORF; this will move through the orf
            kmer = full_sequence[start_index:start_index + k]  # extract the k-mer sequence from the host ORF sequence
            remaining_bases = k  # there will be k remaining bases after the start index that we need to get the coordiantes for. A given k-mer can span multiple sets of coordinates
            kmer_coords = []  # this will store all the coordinate spanned by a given k-mer
            kmer_start_percentile = round((start_index / len(full_sequence)),4)  # we keep track of the percentile in the ORF that a given k-mer is, as it will be useful later.
            original_start_index = start_index # this is the first start index, good to keep track of

            # now, we iterate through each (chr, start, end, strand) 4-tuple in the list 'positions' 
            for chrom, start, end, strand in positions:
                slice_length = end - start + 1  # a given slice has its length determined by the current window of the coordinate we are passing
                
                # we first check to make sure the k-mer starts within this slice
                if start_index < slice_length:
                    # treats plus and minus strand differnetly
                    if strand == "+":
                        slice_start = start + start_index  # we wil now start slicing, according to the current start_index we're in within the for loop
                        slice_end = min(slice_start + remaining_bases - 1, end) # go forward by the number of bases remaining, but make sure we don't yet pass thorugh this current coordinate slice
                    else:  # now comnsideres the reverse strand by reversing how the start and end coordinates are interpreted
                        # we consider the 'end' coord as the de facto start for the revere strand
                        slice_end = end - start_index  # since we're dealing with the reverse strand, the place we slice the orf from the start corresponds to the end position of the ORF slice coords.
                        slice_start = max(slice_end - remaining_bases + 1, start)  # but we make sure to start it no earlier than where the ORF slice's coords
                    
                    # now, since this is a coordinate where the ORF spans, we append it to the coord list for that k-mer
                    kmer_coords.append((chrom, slice_start, slice_end, strand))
                    
                    # we must update the remaining bases to continue processing the coordinates until we get them all
                    remaining_bases -= (slice_end - slice_start + 1)
                    
                    # Reset start_index for subsequent slices
                    start_index = 0  
                    
                    # break if the k-mer is fully mapped
                    if remaining_bases <= 0:
                        break
                else:
                    # reduce start_index to account for bases skipped in this row; i.e., for when we have bases remaining
                    start_index -= slice_length

            # append the result for this k-mer
            results.append({
                "gene": gene,
                "kmer": kmer,
                "coords": kmer_coords,
                "percentile": kmer_start_percentile,
                "position": original_start_index
            })

    # convert the results to a df
    df_results = pd.DataFrame(results)
    return df_results


df_kmers_all = generate_kmers_and_coords(longest_orfs_df_expanded, k=45, offset=3)



# A block of functions for running the feature explorations of k-mers

code = {
    'ATA': 'I', 'ATC': 'I', 'ATT': 'I', 'ATG': 'M',
    'ACA': 'T', 'ACC': 'T', 'ACG': 'T', 'ACT': 'T',
    'AAC': 'N', 'AAT': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGC': 'S', 'AGT': 'S', 'AGA': 'R', 'AGG': 'R',
    'CTA': 'L', 'CTC': 'L', 'CTG': 'L', 'CTT': 'L',
    'CCA': 'P', 'CCC': 'P', 'CCG': 'P', 'CCT': 'P',
    'CAC': 'H', 'CAT': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGA': 'R', 'CGC': 'R', 'CGG': 'R', 'CGT': 'R',
    'GTA': 'V', 'GTC': 'V', 'GTG': 'V', 'GTT': 'V',
    'GCA': 'A', 'GCC': 'A', 'GCG': 'A', 'GCT': 'A',
    'GAC': 'D', 'GAT': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGA': 'G', 'GGC': 'G', 'GGG': 'G', 'GGT': 'G',
    'TCA': 'S', 'TCC': 'S', 'TCG': 'S', 'TCT': 'S',
    'TTC': 'F', 'TTT': 'F', 'TTA': 'L', 'TTG': 'L',
    'TAC': 'Y', 'TAT': 'Y', 'TAA': '*', 'TAG': '*',
    'TGC': 'C', 'TGT': 'C', 'TGA': '*', 'TGG': 'W'}



# counts the number of rare codons in a sequence; given the list of rare codons
def rare_codon_count_0(sequence) -> int:  
    rare_codons = ['TCG', 'CGT', 'ACG', 'CGA', 'CCG', 'GTA', 'CTA', 'GCG', 'ATA','TTA']  # codons with less than 1% frequency in the human ORFeome.

    seq = [sequence[i:i + 3] for i in range(0, len(sequence), 3)]
    # sums the number of rare codons in the seq
    rare_count = sum(codon in rare_codons for codon in seq)
    
    return rare_count
    
def GC(sequence):
    if len(sequence) > 0:
        # counts number of C and G in the sequence
        gc_count = sequence.count('G') + sequence.count('C')
        # gets the proportion of GC in the sequence
        gc_fraction = round(gc_count / len(sequence), 3)
        return gc_fraction
    else:
        return np.nan

def CpG(sequence):
    # ensures sequence is valid
    if len(sequence) > 0:
        # count number of CG dinucloetides
        cpg_count = sequence.count('CG')
        # get prob of CG
        cpg_fraction = round(cpg_count / len(sequence), 3) if len(sequence) > 1 else 0
        return cpg_fraction
    else:
        return np.nan


def shannon_entropy_nuc(sequence):
    length_seq = len(sequence) # gets seq length
    shannon_entropy = 0  # initializes entropy to zero 
    bases = ['A','T','C','G'] # alphabet of bases
    for base in bases:
        p = sequence.count(base)/length_seq  # prob of each base

        if p != 0:  # makes sure p isn't zero to avoid 0log0
            logp = math.log2(p)
            plogp = p*logp  

        else: 
            plogp = 0  # set plogp to zero
        shannon_entropy = shannon_entropy + (plogp) # updates entropy this way
        
    shannon_entropy = -round(shannon_entropy,3) # inverts sign according to SE formula
    return shannon_entropy


# a function to apply each function; creating new cols for each new quantitative feature
# these features are used as part of sampling from the joint distribuition.
def apply_sequence_features(df, seq_col, function_list):
    for func in function_list:
        feature_name = f"{func.__name__}"
        df[feature_name] = df[seq_col].apply(func)
    return df


seq_col = 'kmer'  # the sequence col we wish to apply our functions to
# list of the function names
function_list = [rare_codon_count_0,GC, CpG,shannon_entropy_nuc]

# updates the k-mer df to apply all functions
df_kmers_all = apply_sequence_features(df_kmers_all, seq_col, function_list)

# this may take some time, but varies with CPU speed, RAM, etc.



In [6]:
cfg = loader.config

output_kmers = cfg['outputs']['files']['output_kmers']['path']

## saves kmers to output file
df_kmers_all.to_csv(output_kmers+'df_kmers_all.csv',index=0)

In [7]:

lookup_ORF = dict(zip(longest_orfs_df['name'],longest_orfs_df['assembled_ORF'])) # to lookup the longest orf for each gene; used for processing ORF to merge adjancent k-mers.
df_kmers_copy_human_8_rare = df_kmers_all[df_kmers_all['rare_codon_count_0']>=8] # subsets to 8 or more rare codons per k-mer
df_kmers_copy_human_8_rare = df_kmers_copy_human_8_rare.copy()
df_kmers_copy_human_8_rare['name'] = df_kmers_copy_human_8_rare['gene'].str.split('.').str[0] # cleans the gene name
df_kmers_copy_human_8_rare['full_ORF'] = df_kmers_copy_human_8_rare['name'].map(lookup_ORF) # looks up the longest orf for that gene


# given a list of start coordinates (where each k-mer starts in the ORF), aim is to join together numbers that are close into one interval, 
# so we can use that interval to slice the ORF sequence
def tuple_finder(positions) -> list: # we take a list of positions; i.e., the position that each k-mer starts in the ORF sequence.
    if not positions:
        return []

    result = []  # the result will be a list of tuples
    current_list = [positions[0]]  # we have a working list of a given set of start coords that fall in a 45 nt interval

    for pos in positions[1:]:  # then go through each other position
        if pos - current_list[-1] <= 45: # so long as they are not more than or equal to 45 nt away from the last position
            current_list.append(pos)  # join that position to the current list
        else:  # otherwise, get the first and last position (plus 45, so its the end index)
            result.append((current_list[0], current_list[-1]+45))  
            current_list = [pos]  # now, reset the current list to the current one (since it was outside 45 nt)
    
    # appends the last current_list
    if current_list: 
        result.append((current_list[0], current_list[-1]+45))

    return result  # returns a lit of tuples

# uses the tuple finder to process the k-mers (to merge them)

def process_kmers(kmer_data, tuple_finder):
    # groups the data by the gene name col
    grouped_data = kmer_data.groupby('name')
    # makes  alist of the rows
    row_list = []

    for gene_name, group in grouped_data:
        positions_list = group['position'].tolist()
        # for the list of positions associated with each gene; 
        # i.e., the list where all its rare codon patches start, get their (start,end) from the tuple finder
        meta_kmer_positions = tuple_finder(positions_list)
        # get the sequence of the fulll ORF for that gene
        sequence_ORF = group['full_ORF'].iloc[0]

        # iterates through each (start,end) representing a different merged k-mer slice.
        for start, end in meta_kmer_positions:
            meta_kmer = sequence_ORF[start:end] # gets the merged k-mer; the 'meta_kmer'
            count_rare = rare_codon_count_0(meta_kmer) # counts the number of rare codons 
            prop_rare = count_rare / (len(meta_kmer) / 3)  # proporiton rare codons
            length_in_nt = len(meta_kmer) # length in nt of the k-mer

            row_dict = {
                'ensg_name': gene_name,
                'meta_kmer_id': f"{gene_name}_{(start, end)}", # a unique id we can assign based on the genename, start and end of each k-mer
                'meta_kmer': meta_kmer,
                'count_rare_codons': count_rare,
                'proportion_rare_codons': prop_rare,
                'length_in_nt': length_in_nt,
                'percentile': start / len(sequence_ORF), # gets where the k-mer is in the ORF
                'sequence_ORF_original': sequence_ORF,
            }

            row_list.append(row_dict)

    return pd.DataFrame(row_list) # returns a df of the processed rows


# applies function to generate merged k-mers.
df_kmers_processed = process_kmers(
    kmer_data=df_kmers_copy_human_8_rare, 
    tuple_finder=tuple_finder)



In [8]:


import pandas as pd
import random

# the main goal of this is have a sample space of k-mers that come from the same length distribution of the rare codon patches. 
# to do this, we will generate k-mers from the ORFeome such that they follow the same length distribution as rare codon patches.

# like previous code (function 'generate_kmers_and_coords'), but for each new kmer, it is a random length k from a list of values from the distribution of kmer lengths (of kmers with rare codons.)
# only difference is for each kmer it generates, instead of using k = 45, it choses a random length from the emprical length distribution from rare codon patches

random.seed(1200027) # sets seed
def generate_kmers_and_coords_with_random_lengths(input_df, length_distribution, offset=3):
    results = [] 
    
    grouped = input_df.groupby("gene")

    for gene, group in grouped:
        if group["strand"].iloc[0] == "-":
            group = group.sort_values(by=["end"], ascending=False)  
        else:
            group = group.sort_values(by=["start"])
        
        full_sequence = group["assembled_ORF"].iloc[0]
        positions = []
        for _, row in group.iterrows():
            positions.append((row["chromosome"], row["start"], row["end"], row["strand"]))
        
        start_index = 0  

        # only differnece now is that we're sampling a random length from an emprical distirbution of lengths, rather than a fixed k=45
        while start_index < len(full_sequence):
            k = random.choice(length_distribution) 

            kmer = full_sequence[start_index:start_index + k] 
            if len(kmer) < k: 
                break
            
            remaining_bases = k 
            kmer_coords = [] 
            kmer_start_percentile = round((start_index / len(full_sequence)), 4)
            original_start_index = start_index

            for chrom, start, end, strand in positions:
                row_length = end - start + 1  
                
                if start_index < row_length:
                    if strand == "+":
                        slice_start = start + start_index
                        slice_end = min(slice_start + remaining_bases - 1, end)
                    else:  
                        slice_end = end - start_index  
                        slice_start = max(slice_end - remaining_bases + 1, start) 
                    
                    kmer_coords.append((chrom, slice_start, slice_end, strand))
                    
                    remaining_bases -= (slice_end - slice_start + 1)
                    
                    start_index = 0
                    
                    if remaining_bases <= 0:
                        break
                else:
                    start_index -= row_length

            results.append({
                "gene": gene,
                "kmer": kmer,
                "coords": kmer_coords,
                "percentile": kmer_start_percentile,
                "position": original_start_index
            })

            start_index = original_start_index + offset

    df_results = pd.DataFrame(results)
    return df_results


length_dist = list(df_kmers_processed['length_in_nt']) # defines the emprical length distirbuiton to sample from
df_kmers_all_length = generate_kmers_and_coords_with_random_lengths(longest_orfs_df_expanded, length_dist, offset=3)

In [9]:
df_kmers_all_length['length'] = df_kmers_all_length['kmer'].str.len()

# apply functions to quantify the sequence features of the meta_kmers
seq_col = 'meta_kmer'  
df_kmers_processed = apply_sequence_features(df_kmers_processed, seq_col, function_list)

# also applies it to the length-matched df
seq_col = 'kmer'  
df_kmers_all_length = apply_sequence_features(df_kmers_all_length, seq_col, function_list)



In [ ]:
## can save the length-matched kmers to csv
os.mkdir(output_kmers + 'big_kmers/')
df_kmers_all_length.to_csv(output_kmers + 'big_kmers/' + 'df_kmers_all_length.csv',index=0)

df_kmers_processed.to_csv(output_kmers + 'df_kmers_processed.csv')

In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import gzip 
from Bio import SeqIO
import math
from collections import Counter

# imports for running the k-nearest neigbbors sampling
from scipy.spatial import cKDTree
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy
from scipy.stats import gaussian_kde

import yaml
from pathlib import Path
import pandas as pd

# Simple cd to repo
os.chdir('/Users/jacobfine/Library/CloudStorage/OneDrive-Personal/U of T 2022-2023/Blencowe/Jan_2024_Blencowe/NOV_2024_reanalysis/sequence_analysis_may_2026/compare_backgrounds_codon')
print(f"Working directory: {os.getcwd()}")

# loads files based on config
longest_orfs_df = loader.load_csv('seq_process', 'longest_orfs')
longest_orfs_df_expanded = loader.load_csv('seq_process', 'longest_orfs_expanded')
df_master = loader.load_csv('kmers', 'gene_master')

cfg = loader.config

output_kmers = cfg['outputs']['files']['output_kmers']['path']


output_kmers = cfg['outputs']['files']['output_kmers']['path']

df_kmers_processed = pd.read_csv(output_kmers + 'df_kmers_processed.csv',index_col=0)

df_kmers_all_length = pd.read_csv(output_kmers + 'big_kmers/df_kmers_all_length.csv')

Working directory: /Users/jacobfine/Library/CloudStorage/OneDrive-Personal/U of T 2022-2023/Blencowe/Jan_2024_Blencowe/NOV_2024_reanalysis/sequence_analysis_may_2026/compare_backgrounds_codon
Loaded longest_orfs: (20202, 6)
Loaded longest_orfs_expanded: (199904, 10)
Loaded gene_master: (482, 390)


In [11]:
## the kNN sampling process, using a kd-tree

def sample_rows_with_multiple_matches_kd_sample_genes(
    source_df, target_df, features, meta_kmer_id_col='meta_kmer_id', k_neighbors=7, 
    max_distance=None, random_seed=None, count_vector=None, gene_id_col='ensg_name',seq_col = 'kmer'
):

    if random_seed is not None:
        np.random.seed(random_seed)

    ## defines the source and target features
    source_features = source_df[features].to_numpy() ## all k-mers from all CDSs (or subset)
    target_features = target_df[features].to_numpy() ## rare codon patches


    scaler = StandardScaler()
    source_features_scaled = scaler.fit_transform(source_features) ## standard scaling of all k-mers
    target_features_scaled = scaler.transform(target_features) ## standardizes the rare codon patches to the scale of all k-mers

    # makes KD tree object
    tree = cKDTree(source_features_scaled)

    ## queries k nearest neighbors of the rare codon patches
    distances, indices = tree.query(target_features_scaled, k=k_neighbors)

    # iterates through each rare codon patch and its matches
    results = []
    for target_idx, (matched_indices, dists) in enumerate(zip(indices, distances)):
        if isinstance(matched_indices, int):

            # gets the matches for it
            matched_indices = [matched_indices]
            dists = [dists]
        
        ## for each source kmer matched to a rare codon patch
        for src_idx, dist in zip(matched_indices, dists):
            ## ensure its not too far away in distance
            if max_distance is not None and dist > max_distance:
                continue

            ## append that sampled kmer to the results, with its other info (like distance)
            
            results.append({
                **{col: source_df.iloc[src_idx][col] for col in source_df.columns},
                'matched_from_meta_kmer': target_df.iloc[target_idx][meta_kmer_id_col],
                'distance': dist
            })

    ## makes a df of the results, also ensures the same sequnece is not duplicated

    results_df = pd.DataFrame(results)
    if seq_col in results_df.columns:
        results_df = results_df.drop_duplicates(subset=[seq_col], keep='first')
    
    # sort by distance (closest first)
    results_df = results_df.sort_values('distance')

    # downsamples per gene using count_vector, keeping closest matches (to ensure no gene is overrepresented)
    if count_vector is not None:
        results_df = results_df.groupby(gene_id_col).apply(
            lambda group: group.head(np.random.choice(count_vector))
        ).reset_index(drop=True)

    return results_df

In [13]:
# takes 1 min

rcpgs = list(set(df_kmers_processed['ensg_name'])) # the list of rare codon patch genes


df_kmers_all_length['ensg_name']=df_kmers_all_length['gene'].str.split('.').str[0] # modifies the gene names


In [ ]:

threshold_rare = 2  # common codon threshold (2 or less)

df_kmers_all_length_no_rare = df_kmers_all_length[df_kmers_all_length['rare_codon_count_0']<=threshold_rare]  # select k-mers that have two or less rare codons

df_kmers_all_length_no_rare['is_rare'] = df_kmers_all_length_no_rare['ensg_name'].isin(rcpgs) # also add col to say whether a given k-mer falls in a rare codon patch gene

df_kmers_all_length_no_rare_no_RCPGS = df_kmers_all_length_no_rare[df_kmers_all_length_no_rare['is_rare']==False]  # another df for non-RCPGs, lacking rare patches (for between-genes control)

df_kmers_all_length_no_rare_yes_RCPGS = df_kmers_all_length_no_rare[df_kmers_all_length_no_rare['is_rare']==True]  # another df for only RCPGs, lacking rare patches (for within-genes control)


In [16]:
kmers_per_gene = list(dict(df_kmers_processed['ensg_name'].value_counts()).values()) # gets the kmers per gene

df_kmers_processed['length'] = df_kmers_processed['meta_kmer'].str.len()


In [ ]:
## for R plotting
%load_ext rpy2.ipython

In [ ]:
features = ['GC', 'CpG','shannon_entropy_nuc','length'] ## features to match with


count_vector_rare = list(df_kmers_processed['ensg_name'].value_counts()) ## makes the count vector to downsample from

count_vector_lenient = [2*x for x in count_vector_rare]



within_genes_sample = sample_rows_with_multiple_matches_kd_sample_genes(
    source_df=df_kmers_all_length_no_rare_yes_RCPGS,
    target_df=df_kmers_processed,
    features=features,
    meta_kmer_id_col='meta_kmer_id',
    k_neighbors=10,
    max_distance=1,
    random_seed=18,
    count_vector=count_vector_lenient,
    gene_id_col='ensg_name')

print(f"Within-genes sample: {len(within_genes_sample)} k-mers")


features = ['percentile', 'GC', 'CpG','shannon_entropy_nuc','length']


between_genes_sample = sample_rows_with_multiple_matches_kd_sample_genes(
    source_df=df_kmers_all_length_no_rare_no_RCPGS,
    target_df=df_kmers_processed,
    features=features,
    meta_kmer_id_col='meta_kmer_id',
    k_neighbors=6, 
    max_distance=1,
    random_seed=18,
    count_vector=count_vector_lenient,
    gene_id_col='ensg_name')


print(f"Between-genes sample: {len(between_genes_sample)} k-mers")


In [48]:
longest_orfs_df = loader.load_csv('seq_process', 'longest_orfs') ## for the longest orfs

def CpG(sequence):
    # ensures sequence is valid
    if len(sequence) > 0:
        # count number of CG dinucloetides
        cpg_count = sequence.count('CG')
        # get prob of CG
        cpg_fraction = round(cpg_count / len(sequence), 3) if len(sequence) > 1 else 0
        return cpg_fraction
    else:
        return np.nan

def shannon_entropy_nuc(sequence):
    length_seq = len(sequence) # gets seq length
    shannon_entropy = 0  # initializes entropy to zero 
    bases = ['A','T','C','G'] # alphabet of bases
    for base in bases:
        p = sequence.count(base)/length_seq  # prob of each base

        if p != 0:  # makes sure p isn't zero to avoid 0log0
            logp = math.log2(p)
            plogp = p*logp  

        else: 
            plogp = 0  # set plogp to zero
        shannon_entropy = shannon_entropy + (plogp) # updates entropy this way
        
    shannon_entropy = -round(shannon_entropy,3) # inverts sign according to SE formula
    return shannon_entropy

loaded longest_orfs: (20202, 6)


In [49]:
## for matching full CDSs
longest_orfs_df['ORF_gc'] = longest_orfs_df['assembled_ORF'].apply(GC)

longest_orfs_df['ORF_length'] = longest_orfs_df['assembled_ORF'].str.len()

longest_orfs_df['is_rare'] = longest_orfs_df['name'].isin(rcpgs)
longest_orfs_df_no_rcpgs = longest_orfs_df[longest_orfs_df['is_rare']==False]



In [50]:
def GC(sequence):
    if len(sequence) > 0:
        # counts number of C and G in the sequence
        gc_count = sequence.count('G') + sequence.count('C')
        # gets the proportion of GC in the sequence
        gc_fraction = round(gc_count / len(sequence), 3)
        return gc_fraction
    else:
        return np.nan

In [51]:
df_kmers_processed['ORF_gc'] = df_kmers_processed['sequence_ORF_original'].apply(GC)
df_kmers_processed['ORF_length'] = df_kmers_processed['sequence_ORF_original'].str.len()


In [ ]:
## sample matched genes
features = ['ORF_gc', 'ORF_length']


genes_GC_len_sample = sample_rows_with_multiple_matches_kd_sample_genes(
    source_df=longest_orfs_df_no_rcpgs,
    target_df=df_kmers_processed,
    features=features,
    meta_kmer_id_col='ensg_name',
    k_neighbors=3, 
    max_distance=1,
    random_seed=18,seq_col = 'assembled_ORF')


In [ ]:
%%R -i within_genes_sample,between_genes_sample,df_kmers_processed

library(ggplot2)
library(cowplot)

features <- c('percentile', 'GC', 'CpG','shannon_entropy_nuc','length')
n_bins <- 10  # number of bins for histogram

# fwd KL divergence: KL(P || Q) = KL(rare_patches || sampled distribution)
calculate_kl <- function(rare, sampled, n_bins = 50, eps = 1e-10) {
  rare <- na.omit(rare)
  sampled <- na.omit(sampled)
  
  lower <- min(c(rare, sampled))
  upper <- max(c(rare, sampled))
  breaks <- seq(lower, upper, length.out = n_bins + 1)
  
  p_counts <- hist(rare, breaks = breaks, plot = FALSE)$counts
  q_counts <- hist(sampled, breaks = breaks, plot = FALSE)$counts
  
  p <- p_counts / sum(p_counts)
  q <- q_counts / sum(q_counts)
  
  p <- pmax(p, eps)
  q <- pmax(q, eps)
  
  kl <- sum(p * log(p / q))
  return(kl)
}

# within-genes plots
rare_patches <- df_kmers_processed[, features]
rare_patches$type <- 'rare patches'

background <- within_genes_sample[, features]
background$type <- 'background'

combined_data <- rbind(rare_patches, background)

plot_list <- list()

for (i in 1:length(features)) {
  feat <- features[i]
  kl_val <- calculate_kl(df_kmers_processed[[feat]], within_genes_sample[[feat]], n_bins = n_bins)
  
  p <- ggplot(combined_data, aes_string(x = feat, color = 'type')) +
    geom_density(size = 1) +
    scale_color_manual(values = c('rare patches' = 'firebrick', 'background' = 'steelblue')) +
    theme_classic() +
    labs(title = paste0(feat, '\nKL = ', sprintf('%.3f', kl_val)), x = 'Value', y = 'Density') +
    theme(legend.position = 'none', plot.title = element_text(size = 10))
  plot_list[[i]] <- p
}

p_within <- plot_grid(plotlist = plot_list, ncol = 5, labels = 'AUTO')


ggsave('outputs/figures/figure_s1/background_matching_within_genes_TEST.pdf', p_within, width = 16, height = 3)

# between-genes plots
background <- between_genes_sample[, features]
background$type <- 'background'

combined_data <- rbind(rare_patches, background)

plot_list <- list()

for (i in 1:length(features)) {
  feat <- features[i]
  kl_val <- calculate_kl(df_kmers_processed[[feat]], between_genes_sample[[feat]], n_bins = n_bins)
  
  p <- ggplot(combined_data, aes_string(x = feat, color = 'type')) +
    geom_density(size = 1) +
    scale_color_manual(values = c('rare patches' = 'firebrick', 'background' = 'steelblue')) +
    theme_classic() +
    labs(title = paste0(feat, '\nKL = ', sprintf('%.3f', kl_val)), x = 'Value', y = 'Density') +
    theme(legend.position = 'none', plot.title = element_text(size = 10))
  plot_list[[i]] <- p
}

p_between <- plot_grid(plotlist = plot_list, ncol = 5, labels = 'AUTO')


ggsave('outputs/figures/figure_s1/background_matching_between_genes_TEST.pdf', p_between, width = 16, height = 3)

# save stats for both comparisons
stats_list_within <- list()
stats_list_between <- list()

for (feat in features) {
  rare_mean <- mean(df_kmers_processed[[feat]], na.rm = TRUE)
  rare_sd <- sd(df_kmers_processed[[feat]], na.rm = TRUE)
  
  within_mean <- mean(within_genes_sample[[feat]], na.rm = TRUE)
  within_sd <- sd(within_genes_sample[[feat]], na.rm = TRUE)
  within_kl <- calculate_kl(df_kmers_processed[[feat]], within_genes_sample[[feat]], n_bins = n_bins)
  
  between_mean <- mean(between_genes_sample[[feat]], na.rm = TRUE)
  between_sd <- sd(between_genes_sample[[feat]], na.rm = TRUE)
  between_kl <- calculate_kl(df_kmers_processed[[feat]], between_genes_sample[[feat]], n_bins = n_bins)
  
  stats_list_within[[feat]] <- data.frame(
    Feature = feat,
    Distribution = c('rare patches', 'Within-genes background'),
    Mean = c(rare_mean, within_mean),
    SD = c(rare_sd, within_sd),
    KL_divergence = c(NA, within_kl),
    stringsAsFactors = FALSE
  )
  
  stats_list_between[[feat]] <- data.frame(
    Feature = feat,
    Distribution = c('rare patches', 'Between-genes background'),
    Mean = c(rare_mean, between_mean),
    SD = c(rare_sd, between_sd),
    KL_divergence = c(NA, between_kl),
    stringsAsFactors = FALSE
  )
}

stats_table_within <- do.call(rbind, stats_list_within)
stats_table_between <- do.call(rbind, stats_list_between)
rownames(stats_table_within) <- NULL
rownames(stats_table_between) <- NULL

write.csv(stats_table_within, 'outputs/figures/figure_s1/distribution_statistics_within_genes_TEST.csv', row.names = FALSE)
write.csv(stats_table_between, 'outputs/figures/figure_s1/distribution_statistics_between_genes_TEST.csv', row.names = FALSE)

cat("\nplots and statistics saved to outputs/figures/figure_s1/\n")



In [58]:
%R -o stats_table_between -o stats_table_within

In [62]:

import ast

import ast
import numpy as np

def _parse_coords(val):
    # normalize val into a list of (chrom, start, end, strand) tuples.

    if isinstance(val, (list, np.ndarray)):
        # checks if its already a list
        val = list(val)

    elif isinstance(val, tuple):
        # A single bare coord like ('chr1', 100, 200, '+'); wraps it as a list of 1 item
        val = [val]

    elif isinstance(val, str):
        # if its streing, converts it into a list of roubles

        val = ast.literal_eval(val)
        # also handles case where just one coord is there if no  list
        if isinstance(val, tuple):
            val = [val]

    elif pd.isna(val):
        return []
    return [
        (chrom, float(start), float(end), strand)
        for chrom, start, end, strand in val
    ]


# converts the the df info to bed
def convert_kmers_to_bed(df, output_path):

    
    bed_records = []
    
    for idx, row in df.iterrows():
        coords = _parse_coords(row['coords']) # applies this function to parse the coords
        gene_name = row['ensg_name']
        matched_from = row['matched_from_meta_kmer']
        
        # creates unique name for each kmer 
        unique_name = f"{gene_name}__s{idx}__from__{matched_from}"
        
        # coords is a list of (chrom, start, end, strand) tuples
        for coord_idx, (chrom, start, end, strand) in enumerate(coords):
            # uses cumsum approach: if multiple coords for a kmer (i.e., at boarder of exons)
            if len(coords) > 1:
                unique_name_with_coord = f"{unique_name}__coord{coord_idx}"
            else:
                unique_name_with_coord = unique_name
            
            bed_records.append({
                'chrom': chrom,
                'start': int(start),
                'end': int(end),
                'name': unique_name_with_coord,
                'score': int(row['distance'] * 1000),  # scale distance
                'strand': strand
            })
    
    bed_df = pd.DataFrame(bed_records)
    bed_df = bed_df[['chrom', 'start', 'end', 'name', 'score', 'strand']]
    
    bed_df.to_csv(output_path, sep='\t', header=False, index=False)
    print(f"Saved {len(bed_df)} records to {output_path}")
    
    return bed_df


# converts both samples to bed
within_genes_bed = convert_kmers_to_bed(within_genes_sample, 'outputs/files/output_kmers/within_genes.bed')
between_genes_bed = convert_kmers_to_bed(between_genes_sample, 'outputs/files/output_kmers/between_genes.bed')

# saves original dfs as csv
within_genes_sample.to_csv('outputs/files/output_kmers/within_genes.csv', index=False)
between_genes_sample.to_csv('outputs/files/output_kmers/between_genes.csv', index=False)

print("\nsaved csvs:")
print(f"  within_genes_sample.csv: {len(within_genes_sample)} rows")
print(f"  between_genes_sample.csv: {len(between_genes_sample)} rows")

Saved 571 records to outputs/files/output_kmers/within_genes.bed
Saved 1508 records to outputs/files/output_kmers/between_genes.bed

saved csvs:
  within_genes_sample.csv: 542 rows
  between_genes_sample.csv: 1432 rows


In [68]:
df_kmers_processed

,ensg_name,meta_kmer_id,meta_kmer,count_rare_codons,proportion_rare_codons,length_in_nt,percentile,sequence_ORF_original,rare_codon_count_0,GC,CpG,shannon_entropy_nuc,length,ORF_gc,ORF_length
0,ENSG00000004142,"ENSG00000004142_(117, 168)",TCGCCAGCGTCGACCACGACGACGCGGAGGCACCTCTCGTCCCGAA...,9,0.529412,51,0.105978,ATGGCAGCCTGTACAGCCCGGCGGGCCCTGGCCGTGGGCAGCCGCT...,9,0.686,0.196,1.845,51,0.561,1104
1,ENSG00000004848,"ENSG00000004848_(279, 366)",GGCCGCCTCCTTCAGGGTGCGGCAGCGGCGGCGGCGGCGGCGGCGG...,15,0.517241,87,0.165480,ATGAGCAATCAGTACCAGGAGGAGGGCTGCTCCGAGAGGCCCGAGT...,15,0.874,0.207,1.540,87,0.726,1686
2,ENSG00000005073,"ENSG00000005073_(495, 576)",GAGAAGGGGCCCCCGGCGGCCACGGCGACCTCCGCGGCGGCGGCGG...,13,0.481481,81,0.527157,ATGGATTTTGATGAGCGTGGTCCCTGCTCCTCTAACATGTATTTGC...,13,0.790,0.185,1.726,81,0.617,939
3,ENSG00000006047,"ENSG00000006047_(21, 93)",GCGGGGGCTACAGCGGTCCCCGCGGCGACGGTGCCCGCGACGGCGG...,12,0.500000,72,0.019231,ATGAGCGAGGTGGAGGCGGCAGCGGGGGCTACAGCGGTCCCCGCGG...,12,0.792,0.167,1.698,72,0.671,1092
4,ENSG00000006377,"ENSG00000006377_(111, 189)",CAGCAGCAGCAACAGCAACAGCCGCCGCCGCCGCCGCCGCCGCCGC...,12,0.461538,78,0.126280,ATGATGACCATGACTACGATGGCTGACGGCTTGGAAGGCCAGGACT...,12,0.795,0.154,1.589,78,0.606,879
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,ENSG00000288000,"ENSG00000288000_(93, 150)",CTTCGTGGAGTTCGAGGACTCCCGCGACGCCGACGACGCCGTTTAC...,9,0.473684,57,0.028458,ACGCCTGAGCTACAACGTCCGGGAGAAGGACATCCAGCGCTTTTTC...,9,0.649,0.193,1.935,57,0.584,3268
437,ENSG00000288000,"ENSG00000288000_(162, 243)",CGGCGAGCGCGTGATCGTAGAGCACGCCCGGGGCCCGCGTCGCGAT...,13,0.481481,81,0.049572,ACGCCTGAGCTACAACGTCCGGGAGAAGGACATCCAGCGCTTTTTC...,13,0.728,0.198,1.830,81,0.584,3268
438,ENSG00000288611,"ENSG00000288611_(66, 123)",TGCTCCAACGCGTCGACTCTGGCGCCGCTGCCGGCGCCGCTGGCGG...,9,0.473684,57,0.067073,ATGGACAACGCCTCGTTCTCGGAGCCCTGGCCCGCCAACGCATCGG...,9,0.737,0.158,1.809,57,0.670,984
439,ENSG00000288658,"ENSG00000288658_(549, 600)",CCGCCACCGCCGCCCTCCCCCGCCCCACCGCAGCCGCCGCCGCCGC...,9,0.529412,51,0.530435,ATGGCAGCGCCGGCGAGCGACAGCGGCGGCAGCCAGCAGAGCCCAA...,9,0.902,0.196,1.274,51,0.768,1035


In [ ]:
import yaml
from pathlib import Path
import pandas as pd
## for making a bed file of the rare codon file


# create bed format dataframe with relevant columns
df_bed_rare = df_master[[
    'chrom',
    'start', 
    'end',
    'meta_kmer_id',  # name field in bed
    'count_rare_codons',  # score
    'strand'
]].copy()

# rename columns to match bed format
df_bed_rare.columns = ['chrom', 'start', 'end', 'name', 'score', 'strand']

# ensure correct data types
df_bed_rare['chrom'] = df_bed_rare['chrom'].astype(str)
df_bed_rare['start'] = df_bed_rare['start'].astype(int)
df_bed_rare['end'] = df_bed_rare['end'].astype(int)
df_bed_rare['score'] = df_bed_rare['score'].astype(int)

# save to bed format 
output_dir = Path("outputs/files/output_kmers/")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'rare_patches.bed'


df_bed_rare.to_csv(output_path, sep='\t', header=False, index=False)

print(f"Saved df_bed_rare to {output_path}")
print(f"Shape: {df_bed_rare.shape}")
print(f"\nFirst few rows:")
print(df_bed_rare.head())

Saved df_bed_rare to outputs/files/output_kmers/rare_patches.bed
Shape: (482, 6)

First few rows:
   chrom     start       end                       name  score strand
0  chr17  28355870  28355876  ENSG00000004142_(117_168)      9      -
1   chrX  25013629  25013715  ENSG00000004848_(279_366)     15      -
2   chr7  27184569  27184649  ENSG00000005073_(495_576)     13      -
3  chr17   7294408   7294479    ENSG00000006047_(21_93)     12      -
4   chr7  97006089  97006166  ENSG00000006377_(111_189)     12      +
